In [ ]:
!pip install transformers torch rouge-score nltk

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6724c3bb3b4166564eb73ac0a9930a71d86e7807c1bed48530d2350ced403e98
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
# ===============================================
# Medical Text Simplification using BART
# ===============================================

# 1. Install Required Libraries
# Run this once in Colab
# !pip install transformers torch rouge-score nltk

# ===============================================
# 2. Import Libraries
# ===============================================

import torch
import re
from transformers import BartTokenizer, BartForConditionalGeneration
from rouge_score import rouge_scorer


# ===============================================
# 3. Load Pretrained BART Model
# ===============================================

model_name = "facebook/bart-large"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)


# ===============================================
# 4. Medical Dictionary for Simplification
# ===============================================

medical_dictionary = {

    "hypertension": "high blood pressure",
    "hyperglycemia": "high blood sugar",
    "myocardial infarction": "heart attack",
    "dyspnea": "shortness of breath",
    "analgesic": "pain reliever",
    "edema": "swelling",
    "tachycardia": "fast heart rate",
    "bradycardia": "slow heart rate",
    "intravenous": "given through a vein",
    "antibiotic": "medicine that fights infection"
}


# ===============================================
# 5. Preprocessing Function
# ===============================================

def preprocess_text(text):

    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text


# ===============================================
# 6. Medical Term Replacement
# ===============================================

def replace_medical_terms(text):

    for term in medical_dictionary:
        text = text.replace(term, medical_dictionary[term])

    return text


# ===============================================
# 7. BART Simplification Function
# ===============================================

def simplify_medical_text(text):

    text = preprocess_text(text)

    prompt = """
Rewrite the following medical text in simple language that a patient can understand.
Explain medical terms clearly.

Text:
""" + text

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=40,
        num_beams=5,
        temperature=0.9,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    simplified_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    simplified_text = replace_medical_terms(simplified_text)

    return simplified_text


# ===============================================
# 8. Evaluation Function (ROUGE Score)
# ===============================================

def evaluate(original, simplified):

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    scores = scorer.score(original, simplified)

    return scores


# ===============================================
# 9. Example Medical Texts
# ===============================================

medical_texts = [

    "The patient presents with hypertension and hyperglycemia requiring antihypertensive therapy.",

    "The patient suffered an acute myocardial infarction and requires immediate medical attention.",

    "Dyspnea and tachycardia were observed during physical examination.",

    "Intravenous antibiotics were administered to treat bacterial infection."
]


# ===============================================
# 10. Run Simplification
# ===============================================

for text in medical_texts:

    simplified = simplify_medical_text(text)

    print("\n--------------------------------------")
    print("Original Text:\n", text)
    print("\nSimplified Text:\n", simplified)

    scores = evaluate(text, simplified)

    print("\nROUGE Score:\n", scores)


# ===============================================
# 11. Custom User Input
# ===============================================

while True:

    user_input = input("\nEnter medical text (or type 'exit'): ")

    if user_input.lower() == "exit":
        break

    simplified = simplify_medical_text(user_input)

    print("\nSimplified Output:\n", simplified)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--------------------------------------
Original Text:
 The patient presents with hypertension and hyperglycemia requiring antihypertensive therapy.

Simplified Text:
 Rewrite the following medical text in simple language that a patient can understand.text:the patient presents with high blood pressure and high blood sugar requiring antihypertensive therapy.=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-==-=-=-==-=-==-=-=-==-================================================================================================================================================================================================================================================================================================================================================================================>====>==========

ROUGE Score:
 {'rouge1': Score(precision=0.2857142857142857, recall=0.8, fmeasure=0.4210526315789473), 'rougeL': Score(precision=0.285714285714